In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "07-application-agent-framework/long-running-durable/long-running-agentic/long-running-agents-gcp/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 01 · Durable loop — practice (fill in the blanks)

Implement the two pieces that make the loop durable. Run the check cell after each; it raises with a hint on failure.
Reference solutions: `notebooks/solutions/ex1_durable_loop.py` (peek only after trying).

In [ ]:
import sys, os, json, warnings
warnings.filterwarnings("ignore")
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))          # repo root when run from notebooks/
sys.path[:0] = [os.path.join(ROOT, "src"), os.path.join(ROOT, "notebooks")]

def show_journal(run):
    print(f"run {run.run_id}  status={run.status.value}  version={run.version}  steps={run.usage.steps}  tokens={run.usage.tokens}  cost=${run.usage.cost_usd:.4f}")
    for s in run.journal:
        out = json.dumps(s.output, default=str)[:70] if s.output is not None else (s.error or "")
        print(f"  [{s.index}] {s.kind.value:<6} {s.status.value:<7} {s.name:<18} key={s.idempotency_key or '-':<20} {out}")

In [ ]:
from lragents.core import *
from lragents.practice_checks import check_run_idempotent, check_mini_loop

## Exercise 1 — `run_idempotent`
Execute `fn()` **only if** no result is stored under `key`; return `(result, replayed)`.
Think about the order of *execute* vs *record* and what happens if the process dies between them.

In [ ]:
def run_idempotent(store, key, fn):
    # TODO: if key already in store → return (store.get(key), True)
    # TODO: otherwise run fn(), store.put(key, result), return (result, False)
    raise NotImplementedError

In [ ]:
print(check_run_idempotent(run_idempotent))

## Exercise 2 — a 40-line durable loop
Complete `MiniDurableLoop.step`. Contract:
* If the run is terminal, return it.
* If there is **no pending step**: ask the LLM; journal an `LLM` step (`DONE`); on `final` → set `result`, `SUCCEEDED`, save, return. On a tool call → append a `TOOL` step with status **`STARTED`** and `idempotency_key=f"{run_id}:{index}"`, then **save** (checkpoint #1).
* Execute the pending tool via `run_idempotent` (use your Exercise 1 function or `lragents.core.run_idempotent`), then call `self.faults.maybe_crash("after_side_effect")`, then `finish()` the record and save (checkpoint #2).

In [ ]:
from lragents.core import Run, RunStatus, StepKind, StepRecord, StepStatus, ToolContext, new_run_id

class MiniDurableLoop:
    def __init__(self, store, llm, tools, idem, faults):
        self.store, self.llm, self.tools, self.idem, self.faults = store, llm, tools, idem, faults

    def start(self, goal):
        return self.store.create(Run(run_id=new_run_id("mini"), goal=goal, status=RunStatus.RUNNING))

    def step(self, run_id):
        run = self.store.get(run_id)
        if run.status.terminal:
            return run
        pending = run.pending_step()
        if pending is None:
            d = self.llm.decide("", [{"role": "user", "content": run.goal}], self.tools.specs())
            rec = StepRecord(index=run.next_index(), kind=StepKind.LLM, status=StepStatus.DONE, name="decide")
            rec.finish(output={"kind": d.kind, "tool": d.tool_name, "args": d.tool_args, "text": d.text})
            run.append(rec)
            if d.kind == "final":
                # TODO: set run.result / run.status and return self.store.save(run)
                raise NotImplementedError("TODO: final answer")
            # TODO: append a STARTED TOOL StepRecord (name=d.tool_name, input=d.tool_args, idempotency_key=...)
            # TODO: run = self.store.save(run)   # checkpoint #1 — why here and not after execution?
            # TODO: pending = run.journal[-1]
            raise NotImplementedError("TODO: journal the intent, then checkpoint")
        tool = self.tools.get(pending.name)
        ctx = ToolContext(run.run_id, pending.index, pending.idempotency_key, run.state)
        # TODO: out, replayed = run_idempotent(self.idem, pending.idempotency_key, lambda: tool.fn(pending.input, ctx))
        # TODO: self.faults.maybe_crash("after_side_effect")
        # TODO: pending.finish(output={"result": out, "replayed": replayed}); return self.store.save(run)
        raise NotImplementedError

In [ ]:
print(check_mini_loop(MiniDurableLoop))

## Exercise 3 — reason about it (write answers in the cell below)
1. Your loop saves twice per tool step. Which crash windows does each save close? Which window is still open, and what closes it?
2. Cloud Tasks delivered the same task twice, 50 ms apart, to two Cloud Run instances. Walk through what each instance does with the code you wrote. What primitive is missing from `MiniDurableLoop` that `DurableAgentLoop` has?
3. A tool call takes 45 minutes (a BigQuery export). What changes? (Hint: `is_async`, `WAITING_EVENT`, `schedule_time`.)

In [ ]:
answers = '''
1.
2.
3.
'''

### Reveal the reference implementation

In [ ]:
import inspect
from solutions.ex1_durable_loop import MiniDurableLoop as Ref
print(inspect.getsource(Ref.step))